# مشروع استخراج وتصحيح نصوص الخط اليدوي

### **الخلية 1: تثبيت المكتبات وتجهيز البيئة**
*(قم بتشغيل هذه الخلية أولاً، ستستغرق دقيقتين تقريبًا)*

In [22]:
# 1. تثبيت الأدوات المكتبية والأنظمة الأساسية
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara
# 2. تثبيت مكتبات الذكاء الاصطناعي ومعالجة النصوص
!pip install -q easyocr paddleocr pytesseract transformers torch torchvision fpdf2 Levenshtein langchain langchain_text_splitters
!pip install -q peft huggingface_hub datasets opencv-python-headless langdetect

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-ara is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


### **الخلية 2: إعداد المسارات والاتصال بـ Hugging Face & Drive**

In [23]:
import os, cv2, torch, sqlite3, io, numpy as np, pandas as pd, json, re, base64
from PIL import Image
from google.colab import drive, userdata
from datetime import datetime

# أ. ربط الجوجل درايف
drive.mount('/content/drive')

# ب. إعداد المسارات الموحدة
OUTPUT_DIR = '/content/drive/MyDrive/Handwriting_Dataset'
DB_PATH = os.path.join(OUTPUT_DIR, 'handwriting_data.db')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ج. جلب توكن Hugging Face من Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ تم تحميل Hugging Face Token بنجاح")
except:
    print("⚠️ لم يتم العثور على HF_TOKEN في Secrets. يرجى إضافته يدوياً إذا لزم الأمر.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 الجهاز المستخدم للذكاء الاصطناعي: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ تم تحميل Hugging Face Token بنجاح
🚀 الجهاز المستخدم للذكاء الاصطناعي: cpu


### **الخلية 4: وظائف البحث (القاموس ثنائي اللغة وتصدير Hugging Face)**

### **الخلية 3: محركات الذكاء الاصطناعي (منظومة OCR المتعددة)**

In [25]:
import pytesseract
from paddleocr import PaddleOCR
import easyocr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("⏳ جاري تحميل نماذج الذكاء الاصطناعي (قد يستغرق لحظات)...")

# تهيئة النماذج
try:
    paddle_ocr = PaddleOCR(use_angle_cls=True, lang='ar', show_log=False)
except Exception as e:
    print(f"Failed to load PaddleOCR: {e}")
    paddle_ocr = None
try:
    easy_reader = easyocr.Reader(['ar', 'en'], gpu=torch.cuda.is_available())
except Exception as e:
    print(f"Failed to load EasyOCR: {e}")
    easy_reader = None
try:
    trocr_proc = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
    trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)
except Exception as e:
    print(f"Failed to load TrOCR: {e}")
    trocr_proc, trocr_model = None, None

def enhance_digit_recognition(text):
    """تصحيح الأرقام المتداخلة مع الحروف (بناءً على طلبك دكتور)"""
    if not text: return text
    corrections = {'O':'0', 'o':'0', 'I':'1', 'l':'1', '|':'1', 'Z':'2', 'z':'2',
                   'S':'5', 's':'5', 'G':'6', 'g':'6', 'T':'7', 't':'7', 'B':'8', 'b':'8'}
    for wrong, correct in corrections.items():
        text = text.replace(wrong, correct)
    return text

def get_ensemble_predictions(image_bytes):
    """تجميع نتائج 4 نماذج مختلفة لضمان أعلى دقة"""
    nparr = np.frombuffer(image_bytes, np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    results = []

    # 1. TrOCR
    if trocr_model and trocr_proc:
        try:
            pixel_values = trocr_proc(images=pil_img, return_tensors="pt").pixel_values.to(device)
            out = trocr_model.generate(pixel_values)
            results.append(trocr_proc.batch_decode(out, skip_special_tokens=True)[0])
        except: pass

    # 2. Tesseract
    try: results.append(pytesseract.image_to_string(pil_img, config='--psm 7').strip())
    except: pass

    # 3. PaddleOCR
    if paddle_ocr:
        try:
            p_res = paddle_ocr.ocr(img_bgr, cls=False)
            if p_res and p_res[0]: results.append(" ".join([l[1][0] for l in p_res[0]]))
        except: pass

    # 4. EasyOCR
    if easy_reader:
        try:
            e_res = easy_reader.readtext(img_bgr)
            results.append(" ".join([it[1] for it in e_res]))
        except: pass

    unique = list(dict.fromkeys([enhance_digit_recognition(r) for r in results if r]))
    return unique[:3]

RuntimeError: PDX has already been initialized. Reinitialization is not supported.

### **الخلية 3: تحسين الأرقام ودوال التصدير (JSONL + Hugging Face)**

In [ ]:
from huggingface_hub import HfApi, login

def extract_bilingual_vocab():
    """استخراج المفردات الطبية والهندسية من البيانات التي قمت بتصحيحها"""
    with sqlite3.connect(DB_PATH) as conn:
        words = pd.read_sql_query("SELECT * FROM handwriting_data WHERE status='verified' ORDER BY page_num, y, x", conn)

    if words.empty: return "⚠️ لا توجد كلمات موثقة كـ 'verified' حالياً."

    vocab_pairs = []
    # (المنطق الذي طلبته لاستخراج الثنائيات)
    for page in words['page_num'].unique():
        p_words = words[words['page_num'] == page]
        texts = p_words['predicted_text'].astype(str).tolist()
        en = [t for t in texts if all(ord(c) < 128 for c in t)]
        ar = [t for t in texts if any('\u0600' <= c <= '\u06FF' for c in t)]
        if en and ar:
            vocab_pairs.append({'english': ' | '.join(en[:3]), 'arabic': ' | '.join(ar[:3]), 'page': page})

    df = pd.DataFrame(vocab_pairs)
    df.to_csv(os.path.join(OUTPUT_DIR, 'bilingual_vocab.csv'), index=False, encoding='utf-8-sig')
    return df

def upload_to_hf(repo_name="handwriting-ocr-dataset"):
    """تصدير JSONL ورفعه مباشرة لحسابك"""
    login(token=os.environ.get('HF_TOKEN')) # Ensure login using token from env var
    jsonl_path = os.path.join(OUTPUT_DIR, "data.jsonl")
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("SELECT image_data, predicted_text FROM handwriting_data WHERE status='verified'", conn)

    with open(jsonl_path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            img_b64 = base64.b64encode(row['image_data']).decode('utf-8')
            f.write(json.dumps({"image": img_b64, "text": row['predicted_text']}, ensure_ascii=False) + '\n')

    api = HfApi()
    user = userdata.get('HF_USERNAME', 'dr_abdulmalek') # Using default if not set
    api.upload_file(path_or_fileobj=jsonl_path, path_in_repo="data.jsonl", repo_id=f"{user}/{repo_name}", repo_type="dataset")
    print(f"🚀 تم الرفع بنجاح!")

### **الخلية 5: الواجهة الرسومية التفاعلية (المحرك الرئيسي)**

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# تهيئة قاعدة البيانات والتخزين المؤقت
conn = sqlite3.connect(DB_PATH)
try: conn.execute("ALTER TABLE handwriting_data ADD COLUMN cached_predictions TEXT")
except: pass
df = pd.read_sql_query("SELECT * FROM handwriting_data ORDER BY image_id", conn)
current_idx = 0

# بناء الواجهة
img_disp = widgets.Image(format='png', width=500)
text_input = widgets.Text(description="التصحيح:", layout={'width': '95%'})
info_lbl = widgets.Label()
btns = [widgets.Button(layout={'width': '32%'}, button_style='warning') for _ in range(3)]

def update_ui():
    global current_idx
    if current_idx < len(df):
        row = df.iloc[current_idx]
        img_disp.value = row['image_data']

        # استخدام التخزين المؤقت لتوفير وقت المعالجة
        if row['cached_predictions'] and pd.notna(row['cached_predictions']):
            preds = json.loads(row['cached_predictions'])
        else:
            preds = get_ensemble_predictions(row['image_data'])
            with sqlite3.connect(DB_PATH) as c:
                c.execute("UPDATE handwriting_data SET cached_predictions=? WHERE image_id=?", (json.dumps(preds), int(row['image_id'])))

        for i in range(3):
            if i < len(preds): btns[i].description = preds[i]; btns[i].disabled = False
            else: btns[i].disabled = True
        text_input.value = preds[0] if preds else ""
        info_lbl.value = f"صورة رقم: {row['image_id']} | التقدم: {current_idx+1}/{len(df)}"

def on_next(_):
    global current_idx
    with sqlite3.connect(DB_PATH) as c:
        c.execute("UPDATE handwriting_data SET predicted_text=?, status='verified' WHERE image_id=?", (text_input.value, int(df.iloc[current_idx]['image_id'])))
    current_idx += 1
    update_ui()

next_btn = widgets.Button(description="التالي ➡️", button_style='success')
next_btn.on_click(on_next)
for b in btns: b.on_click(lambda x: [setattr(text_input, 'value', x.description), on_next(None)])

# أزرار البحث والتصدير
vocab_btn = widgets.Button(description="استخراج القاموس 📖", button_style='info')
hf_btn = widgets.Button(description="رفع لـ HuggingFace 🤗", button_style='primary')
vocab_btn.on_click(lambda x: display(extract_bilingual_vocab()))
hf_btn.on_click(lambda x: upload_to_hf())

display(widgets.VBox([
    widgets.HTML("<h2>🩺 منظومة د. عبد الملك للأرشفة والذكاء الاصطناعي</h2>"),
    info_lbl, img_disp, widgets.HBox(btns), text_input,
    widgets.HBox([next_btn, vocab_btn, hf_btn])
]))
update_ui()

### **(الخلية القديمة) تم استبدالها: نموذج الأرقام الاختياري**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **الخلية 6: التدريب الليلي (Fine-tuning)**
*(شغل هذه الخلية فقط عندما تريد تدريب الموديل على بياناتك المصححة)*

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, default_data_collator, TrOCRProcessor, VisionEncoderDecoderModel
from datasets import load_dataset

# هذا هو كود التدريب الكامل (Full Fine-tuning) الذي ناقشناه سابقاً.
# يُنصح بتشغيله بعد جمع 200+ عينة مصادقة (verified) على الأقل.

DATASET_REPO = "your-username/handwriting-ocr-dataset"  # غيّر إلى اسم المستودع الخاص بك على Hugging Face
OUTPUT_FT = "/content/drive/MyDrive/Handwriting_Dataset/trocr_full_finetuned"

def decode_image(example):
    # الدالة تقوم بتحويل الصورة من base64 إلى PIL Image
    example['image'] = Image.open(io.BytesIO(base64.b64decode(example['image']))).convert("RGB")
    return example

# تحقق مما إذا كان هناك HF_TOKEN موجود قبل محاولة تحميل مجموعة البيانات
if os.environ.get('HF_TOKEN'):
    try:
        # يجب التأكد من أنك قمت برفع dataset إلى Hugging Face أولاً باستخدام وظيفة `upload_to_hf`
        dataset = load_dataset(DATASET_REPO, split='train', data_files="data.jsonl", token=os.environ.get('HF_TOKEN'))
        dataset = dataset.map(decode_image)
        dataset = dataset.train_test_split(test_size=0.1)

        processor_ft = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
        model_ft = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)

        def preprocess_ft(examples):
            pixel_values = processor_ft(images=examples['image'], return_tensors="pt").pixel_values
            labels = processor_ft.tokenizer(examples['text'], padding="max_length", max_length=128, truncation=True).input_ids
            labels = [[l if l != processor_ft.tokenizer.pad_token_id else -100 for l in label] for label in labels]
            return {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels).squeeze()}

        train_ds = dataset["train"].map(preprocess_ft, batched=True, remove_columns=dataset["train"].column_names)
        eval_ds = dataset["test"].map(preprocess_ft, batched=True, remove_columns=dataset["test"].column_names)

        training_args = Seq2SeqTrainingArguments(
            output_dir=OUTPUT_FT, predict_with_generate=True, evaluation_strategy="epoch",
            per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-5,
            num_train_epochs=10, fp16=True, logging_steps=10, save_strategy="epoch"
        )
        trainer = Seq2SeqTrainer(
            model=model_ft, args=training_args, train_dataset=train_ds, eval_dataset=eval_ds,
            data_collator=default_data_collator, tokenizer=processor_ft.feature_extractor
        )
        trainer.train()
        model_ft.save_pretrained(OUTPUT_FT)
        processor_ft.save_pretrained(OUTPUT_FT)
        print(f"🏁 اكتمل التدريب الكامل – النموذج في {OUTPUT_FT}")
    except Exception as e:
        print(f"⚠️ حدث خطأ أثناء إعداد أو تشغيل التدريب الكامل: {e}")
        print("يرجى التأكد من أن مجموعة البيانات قد تم رفعها إلى Hugging Face بشكل صحيح وأن الـ HF_TOKEN صالح.")
else:
    print("⚠️ لا يمكن بدء التدريب الكامل. Hugging Face Token غير متوفر.")
print("جاهز لبدء التدريب الكامل فور توفر عينات كافية.")

**ملاحظات دكتور:**
 1. **التخزين المؤقت:** أضفت خاصية cached_predictions في الخلية 5 لضمان أن النظام لا يكرر معالجة الصورة مرتين، مما يجعل التنقل سريعاً جداً.
 2. **تحسين الأرقام:** دمجت دالة enhance_digit_recognition ضمن تدفق المعالجة الأساسي.
 3. **القاموس:** دالة extract_bilingual_vocab أصبحت الآن زرًا داخل الواجهة، تضغط عليه في أي وقت ليعرض لك القاموس المستخرج من عملك.

### **(الخلية القديمة) تم استبدالها: تحميل النماذج**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) تم استبدالها: دوال المعالجة**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) تم استبدالها: دالة معالجة PDF**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) تم استبدالها: تشغيل الاستخراج**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) تم استبدالها: الواجهة المتكاملة**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **الخلية 8: عرض مسارات ملفات المراقبة (للإشارة)**

In [ ]:
print("\n📁 ملفات المراقبة والتطوير:")
print(f"   • سجل الأحداث: {LOG_FILE}")
print(f"   • إحصائيات المعالجة: {STATS_JSON}")
print(f"   • تصحيحات المستخدم: {FEEDBACK_CSV}")

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) حالة التحديث**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) تم استبدالها: واجهة مراجعة الجمل**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass

### **(الخلية القديمة) قسم معاد تنظيمه**

In [ ]:
# هذه الخلية لم تعد مستخدمة في الهيكل الجديد للمشروع.
pass